In [1]:
# Install py_stringmatching if it's not already present in the environment
!pip install py_stringmatching

In [2]:
# Importing the libraries required for all tasks
import re
import csv
import time
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler as mms
from sklearn.preprocessing import StandardScaler as ss
from py_stringmatching import similarity_measure as sm

# Task 1: Profiling relational data

For this task, we used the file [Road Safety Data - Collisions - 2024](https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-collision-2024.csv) from the [road safety dataset](https://www.data.gov.uk/dataset/cb7ae6f0-4be6-4935-9277-47e5ce24a11f/road-safety-data). In the cells below, we compute the following summary statistics on the dataset:
1. Number of rows & columns – shows dataset size and complexity.
2. Data types – indicates whether attributes are numeric, text, or date/time.
3. Missing values – highlights completeness issues and cleaning needs.
4. Distinct values – shows attribute variability and possible keys.
5. Uniqueness ratio – quickly detects potential identifiers.
6. Mode (most frequent value) – shows common/default entries.
7. Constancy – reveals columns dominated by a single value (maybe not useful).
8. Duplicate rows – identifies redundancy that can distort analysis.
9. Minimum & Maximum – reveal range and detect outliers.
10. Mean, Median, Standard Deviation – describe central tendency and spread; help understand skew and variability. 

In [3]:
# Load the dataset 
df = pd.read_csv("dft-road-casualty-statistics-collision-2024.csv")

/var/folders/mh/2564dbl57pbb8nm7fh_7gn540000gn/T/ipykernel_19896/3581080808.py:2: DtypeWarning: Columns (0,2,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dft-road-casualty-statistics-collision-2024.csv")


#### 1. Number of rows and columns 

In [4]:
rows, cols = df.shape 
print(f"Rows: {rows}, Columns: {cols}")

Rows: 100927, Columns: 44


#### 2. Data types per column 

In [5]:
dtypes = df.dtypes
print("Data types:\n", dtypes) 

Data types:
 collision_index                                      object
collision_year                                        int64
collision_ref_no                                     object
location_easting_osgr                                 int64
location_northing_osgr                                int64
longitude                                           float64
latitude                                            float64
police_force                                          int64
collision_severity                                    int64
number_of_vehicles                                    int64
number_of_casualties                                  int64
date                                                 object
day_of_week                                           int64
time                                                 object
local_authority_district                              int64
local_authority_ons_district                         object
local_authority_highway    

#### 3. Missing values per column 

In [6]:
missing = df.isna().sum()
print("Missing values:\n", missing)

Missing values:
 collision_index                                     0
collision_year                                      0
collision_ref_no                                    0
location_easting_osgr                               0
location_northing_osgr                              0
longitude                                           0
latitude                                            0
police_force                                        0
collision_severity                                  0
number_of_vehicles                                  0
number_of_casualties                                0
date                                                0
day_of_week                                         0
time                                                0
local_authority_district                            0
local_authority_ons_district                        0
local_authority_highway                             0
local_authority_highway_current                     3
first_road_

#### 4. Distinct values per column 

In [7]:
distinct = df.nunique()
print("Distinct values:\n", distinct) 

Distinct values:
 collision_index                                     100927
collision_year                                           1
collision_ref_no                                    100927
location_easting_osgr                                82639
location_northing_osgr                               83251
longitude                                            87349
latitude                                             81912
police_force                                            44
collision_severity                                       3
number_of_vehicles                                      15
number_of_casualties                                    17
date                                                   366
day_of_week                                              7
time                                                  1439
local_authority_district                                 1
local_authority_ons_district                           351
local_authority_highway               

#### 5. Uniqueness ratio (distinct / total rows) 

In [8]:
uniqueness = distinct / rows 
print("Uniqueness ratio:\n", uniqueness) 

Uniqueness ratio:
 collision_index                                     1.000000
collision_year                                      0.000010
collision_ref_no                                    1.000000
location_easting_osgr                               0.818800
location_northing_osgr                              0.824864
longitude                                           0.865467
latitude                                            0.811597
police_force                                        0.000436
collision_severity                                  0.000030
number_of_vehicles                                  0.000149
number_of_casualties                                0.000168
date                                                0.003626
day_of_week                                         0.000069
time                                                0.014258
local_authority_district                            0.000010
local_authority_ons_district                        0.003478
local

#### 6. Most frequent value (mode) per column 

In [9]:
modes = df.mode().iloc[0] 
print("Mode:\n", modes) 

Mode:
 collision_index                                     202417H103224
collision_year                                             2024.0
collision_ref_no                                        17H103224
location_easting_osgr                                    532827.0
location_northing_osgr                                   169892.0
longitude                                                -0.17161
latitude                                                 51.38721
police_force                                                  1.0
collision_severity                                            3.0
number_of_vehicles                                            2.0
number_of_casualties                                          1.0
date                                                   19/07/2024
day_of_week                                                   6.0
time                                                        17:00
local_authority_district                                     -1.0
loc

/var/folders/mh/2564dbl57pbb8nm7fh_7gn540000gn/T/ipykernel_19896/639248812.py:1: UserWarning: Unable to sort modes: '<' not supported between instances of 'int' and 'str'
  modes = df.mode().iloc[0]
/var/folders/mh/2564dbl57pbb8nm7fh_7gn540000gn/T/ipykernel_19896/639248812.py:1: UserWarning: Unable to sort modes: '<' not supported between instances of 'int' and 'str'
  modes = df.mode().iloc[0]


#### 7. Constancy (frequency of most common value / total rows) 

In [10]:
constancy = df.apply(lambda c: c.value_counts(dropna=True).max()/rows) 
print("Constancy:\n", constancy) 

Constancy:
 collision_index                                     0.000010
collision_year                                      1.000000
collision_ref_no                                    0.000010
location_easting_osgr                               0.000079
location_northing_osgr                              0.000069
longitude                                           0.000069
latitude                                            0.000079
police_force                                        0.206426
collision_severity                                  0.751613
number_of_vehicles                                  0.616525
number_of_casualties                                0.815233
date                                                0.004013
day_of_week                                         0.162930
time                                                0.009809
local_authority_district                            1.000000
local_authority_ons_district                        0.022749
local_author

#### 8. Duplicate rows count  

In [11]:
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates) 

Duplicate rows: 0


#### 9. Minimum and Maximum for numeric columns

In [12]:
 num = df.select_dtypes(include=[np.number]) 
num_min = num.min() 
num_max = num.max()
print("Numeric min:\n", num_min) 
print("Numeric max:\n", num_max) 

Numeric min:
 collision_year                                       2024.00000
location_easting_osgr                               75390.00000
location_northing_osgr                              10211.00000
longitude                                              -7.38868
latitude                                               49.91221
police_force                                            1.00000
collision_severity                                      1.00000
number_of_vehicles                                      1.00000
number_of_casualties                                    1.00000
day_of_week                                             1.00000
local_authority_district                               -1.00000
first_road_class                                        1.00000
first_road_number                                       0.00000
road_type                                               1.00000
speed_limit                                            -1.00000
junction_detail_historic  

#### 10. Mean, Median, Standard deviation for numeric columns 

In [13]:
num_mean = num.mean() 
num_median = num.median() 
num_std = num.std() 
print("Numeric mean:\n", num_mean)
print("Numeric median:\n", num_median) 
print("Numeric std:\n", num_std) 

Numeric mean:
 collision_year                                        2024.000000
location_easting_osgr                               453683.344348
location_northing_osgr                              277289.171441
longitude                                               -1.229156
latitude                                                52.383155
police_force                                            27.872730
collision_severity                                       2.736731
number_of_vehicles                                       1.818285
number_of_casualties                                     1.270938
day_of_week                                              4.123842
local_authority_district                                -1.000000
first_road_class                                         4.234120
first_road_number                                      788.761303
road_type                                                5.261020
speed_limit                                             35.87

# Task 2: Entity resolution

## Task 2 - Part 1: Pairwise comparison of all records
In the below cells, we compare every single record in the dataset (ACM.csv) with all the records in (DBLP2.csv) to find the similar records (records that represent the same publication) using the following steps:
1. Normalize all records
    - Change all alphabetical characters into lowercase.
    - Convert multiple spaces to one.
2. Define functions to compute the following similarity measures:
    - Levenshtein distance & similarity (for title)
    - Jaro similarity (for author)
    - Modified Affine gap similarity (for venue)
    - Match (1) / Mismatch (0) (for year)
3. Compute record similarity
    - Using the formula $rec\_sim = w_1*s_t + w_2*s_a + w_3*s_c + w_4*s_y$ where $\sum_{i=1}^4 w_i = 1$
    - Report the records with rec_sim > 0.7 as duplicate records by storing the ids of both records in a list
    - Report the running time of the method
4. Compute the precision of this method

In [14]:
# Load the datasets
df1 = pd.read_csv(filepath_or_buffer = 'ACM.csv',
                 delimiter=',', doublequote=True, quotechar='"',
                 na_values = ['na', '-', '.', ''], encoding = 'ISO-8859-1')
df2 = pd.read_csv(filepath_or_buffer = 'DBLP2.csv',
                 delimiter=',', doublequote=True, quotechar='"',
                 na_values = ['na', '-', '.', ''], encoding='ISO-8859-1')
df3 = pd.read_csv(filepath_or_buffer = 'DBLP-ACM_perfectMapping.csv',
                 delimiter=',', doublequote=True, quotechar='"',
                 na_values = ['na', '-', '.', ''], encoding='ISO-8859-1')

#### 1. Normalize all records:
- Change all alphabetical characters into lowercase.
- Convert multiple spaces to one.

In [15]:
# Normalization function: lowercase, strip, collapse multiple spaces
normalize = lambda s: re.sub(r"\s+", " ", str(s)).strip().lower()

# Apply to ACM table
for col in ["title", "authors", "venue","year"]:
    df1[col] = df1[col].apply(normalize)

# Apply to DBLP2 table
for col in ["title", "authors", "venue","year"]:
    df2[col] = df2[col].apply(normalize)

#### 2. Define functions to compute the following similarity measures:
- Levenshtein distance & similarity
- Jaro similarity
- Modified Affine gap similarity
- Match (1) / Mismatch (0)

In [16]:
# --- Levenshtein distance & similarity ---
def ldist(s, t):
    rows, cols = len(s)+1, len(t)+1
    dist = [[0]*cols for _ in range(rows)]
    for i in range(1, rows):
        dist[i][0] = i
    for j in range(1, cols):
        dist[0][j] = j
    for i in range(1, rows):
        for j in range(1, cols):
            cost = 0 if s[i-1] == t[j-1] else 1
            dist[i][j] = min(dist[i-1][j]+1,
                             dist[i][j-1]+1,
                             dist[i-1][j-1]+cost)
    return dist[-1][-1]

def lev_sim(s1, s2):
    if not s1 and not s2: return 1.0
    return 1 - ldist(str(s1), str(s2)) / max(len(str(s1)), len(str(s2)))

# --- Jaro similarity ---
def jaro_sim(s1, s2):
    if pd.isna(s1) and pd.isna(s2): return 1.0
    if pd.isna(s1) or pd.isna(s2): return 0.0
    jaro = sm.jaro.Jaro()
    return jaro.get_raw_score(str(s1), str(s2))

# --- Affine gap similarity ---
def aff_sim(s1, s2, gap_start=1, gap_ext=0.1):
    if pd.isna(s1) and pd.isna(s2): return 1.0
    if pd.isna(s1) or pd.isna(s2): return 0.0
    aff = sm.affine.Affine(
        gap_start=gap_start, gap_continuation=gap_ext,
        sim_func=lambda a, b: 1 if a == b else 0
    )
    raw_score = aff.get_raw_score(str(s1), str(s2))
    max_len = max(len(str(s1)), len(str(s2)))
    return raw_score / max_len if max_len > 0 else 1.0

# --- Year match ---
def year_match(y1, y2):
    return 1 if y1 == y2 else 0

#### 3. Compute record similarity
- Using the formula $rec\_sim = w_1*s_t + w_2*s_a + w_3*s_c + w_4*s_y$ where $\sum_{i=1}^4 w_i = 1$
- Report the records with rec_sim > 0.7 as duplicate records by storing the ids of both records in a list
- Report the running time of the method

In [17]:
# Function to compute record similarity
def record_similarity(row1, row2, w1=0.4, w2=0.3, w3=0.2, w4=0.1):
    st = lev_sim(row1["title"], row2["title"])      # title
    sa = jaro_sim(row1["authors"], row2["authors"]) # authors
    sc = aff_sim(row1["venue"], row2["venue"])      # venue
    sy = year_match(row1["year"], row2["year"])     # year
    rec_sim = w1*st + w2*sa + w3*sc + w4*sy
    return rec_sim

# Computing record similarity and noting the running time of the method
pairwise_comp_start_time = time.time()
results = []
for i, r1 in df1.iterrows():
    for j, r2 in df2.iterrows():
        sim = record_similarity(r1, r2)
        if sim > 0.7:
            results.append((r1["id"], r2["id"], sim))
pairwise_comp_end_time = time.time()
pairwise_comp_runtime = pairwise_comp_end_time - pairwise_comp_start_time

predicted_df = pd.DataFrame(results, columns=["id1", "id2", "rec_sim"])
print(predicted_df)
print(f"Running Time: {pairwise_comp_runtime:.2f} seconds")

         id1                          id2   rec_sim
0     304586        conf/sigmod/VossenW99  0.728004
1     304587          conf/sigmod/CruzJ99  0.743254
2     304589  conf/sigmod/BouguettayaBH99  0.745718
3     304590     conf/sigmod/BaruGLMPVC99  0.766210
4     304582     conf/sigmod/BrodskySCE99  0.793906
...      ...                          ...       ...
2268  672977          conf/vldb/KemperK94  0.787619
2269  950482  journals/vldb/BernsteinIR03  0.720482
2270  672980           conf/vldb/Guting94  0.757356
2271  945741   journals/sigmod/Anisimov03  0.935050
2272  672979          conf/vldb/WienerN94  0.725498

[2273 rows x 3 columns]
Running Time: 3048.07 seconds


#### 4. Compute the precision of this method

In [19]:
true_pairs = set((str(x[1]).strip(), str(x[0]).strip()) for x in df3.values)

# Predicted set
predicted_set = set((str(a), str(b)) for a, b, _ in results)

# Intersection
correct = predicted_set & true_pairs

# Precision
precision = len(correct) / len(predicted_set) if predicted_set else 0

print("Correctly found:", len(correct))
print("Precision:", precision)

Correctly found: 2085
Precision: 0.9172899252089749


## Task 2 - Part 2: Finding similar documents using LSH

In the cells below, we apply Locality Sensitive Hashing (LSH) for finding similar documents using the steps below:
1. Concatenate the values in each record into one single string.
2. Change all alphabetical characters into lowercase.
3. Convert multiple spaces to one.
4. Combine the records from both tables into one big list as we did during the lab.
5. Use the functions in the tutorials from lab 5 to compute the shingles, the minhash signature and the similarity.
6. Compute record similarity using LSH algorithm
   - Extract the top 2224 candidates from the LSH algorithm
   - Compare them to the actual mappings in the file DBLP-ACM_perfectMapping.csv
   - Compute the precision of the method.
   - Record the running time of the method.
7. Compare the precision and the running time in Parts 1 and 2. 

In [20]:
# Load the datasets again (since they were modified in Task 2 - Part 1)
acm_df = pd.read_csv(filepath_or_buffer = 'ACM.csv',
                 delimiter=',', doublequote=True, quotechar='"',
                 na_values = ['na', '-', '.', ''], encoding = 'ISO-8859-1')
dblp2_df = pd.read_csv(filepath_or_buffer = 'DBLP2.csv',
                 delimiter=',', doublequote=True, quotechar='"',
                 na_values = ['na', '-', '.', ''], encoding='ISO-8859-1')
dblpacm_df = pd.read_csv(filepath_or_buffer = 'DBLP-ACM_perfectMapping.csv',
                 delimiter=',', doublequote=True, quotechar='"',
                 na_values = ['na', '-', '.', ''], encoding='ISO-8859-1')

#### 1. Concatenate the values in each record into one single string.

In [21]:
row_strings1 = acm_df.astype(str).agg(' '.join, axis=1)
row_strings2 = dblp2_df.astype(str).agg(' '.join, axis=1)

#### 2. Change all alphabetical characters into lowercase.

In [22]:
row_string1 = row_strings1.str.lower()
row_string2 = row_strings2.str.lower()

#### 3. Convert multiple spaces to one.

In [23]:
row_string1 = row_string1.apply(lambda x: ' '.join(x.split()))
row_string2 = row_string2.apply(lambda x: ' '.join(x.split()))

#### 4. Combine the records from both tables into one big list as we did during the lab.

In [24]:
one_big_list = row_string1.tolist() + row_string2.tolist()

#### 5. Use the functions in the tutorials from lab 5 to compute the shingles, the minhash signature and the similarity.

In [25]:
def shingle(text: str, k: int)->set:
    """
    Create a set of 'shingles' from the input text using k-shingling.

    Parameters:
        text (str): The input text to be converted into shingles.
        k (int): The length of the shingles (substring size).

    Returns:
        set: A set containing the shingles extracted from the input text.
    """
    shingle_set = []
    for i in range(len(text) - k+1):
        shingle_set.append(text[i:i+k])
    return set(shingle_set)

def build_vocab(shingle_sets: list)->dict:
    """
    Constructs a vocabulary dictionary from a list of shingle sets.

    This function takes a list of shingle sets and creates a unified vocabulary
    dictionary. Each unique shingle across all sets is assigned a unique integer
    identifier.

    Parameters:
    - shingle_sets (list of set): A list containing sets of shingles.

    Returns:
    - dict: A vocabulary dictionary where keys are the unique shingles and values
      are their corresponding unique integer identifiers.

    Example:
    sets = [{"apple", "banana"}, {"banana", "cherry"}]
    build_vocab(sets)
    {'apple': 0, 'cherry': 1, 'banana': 2}  # The exact order might vary due to set behavior
    """

    # Builds a single set containing every unique shingle that appears in any document.
    full_set = set()
    for set_ in shingle_sets:
      for item in set_:
        full_set.add(item)

    # contains all unique shingles across all documents - {<element from full_set>: <index from full_set>}
    vocab = {}
    for i, shingle in enumerate(list(full_set)):
        vocab[shingle] = i
    return vocab

def one_hot(shingles: set, vocab: dict):
    vec = np.zeros(len(vocab)) # create a vector of zeros
    for shingle in shingles:
        idx = vocab[shingle]
        vec[idx] = 1
    return vec


def get_minhash_arr(num_hashes:int,vocab:dict):
    """
    Generates a MinHash array for the given vocabulary.

    This function creates an array where each row represents a hash function and
    each column corresponds to a word in the vocabulary. The values are permutations
    of integers representing the hashed value of each word for that particular hash function.

    Parameters:
    - num_hashes (int): The number of hash functions (rows) to generate for the MinHash array.
    - vocab (dict): The vocabulary where keys are words and values can be any data
      (only keys are used in this function).

    Returns:
    - np.ndarray: The generated MinHash array with `num_hashes` rows and columns equal
      to the size of the vocabulary. Each cell contains the hashed value of the corresponding
      word for the respective hash function.

    Example:
    vocab = {'apple': 1, 'banana': 2}
    get_minhash_arr(2, vocab)
    # Possible output:
    # array([[1, 2],
    #        [2, 1]])
    """
    length = len(vocab.keys())
    arr = np.zeros((num_hashes,length))
    for i in range(num_hashes):
        permutation = np.random.permutation(len(vocab.keys())) + 1
        arr[i,:] = permutation.copy()
    return arr.astype(int)


def get_signature(minhash:np.ndarray, vector:np.ndarray):
    """
    Computes the signature of a given vector using the provided MinHash matrix.

    The function finds the nonzero indices of the vector, extracts the corresponding
    columns from the MinHash matrix, and computes the signature as the minimum value
    across those columns for each row of the MinHash matrix.

    Parameters:
    - minhash (np.ndarray): The MinHash matrix where each column represents a shingle
      and each row represents a hash function.
    - vector (np.ndarray): A vector representing the presence (non-zero values) or
      absence (zero values) of shingles.

    Returns:
    - np.ndarray: The signature vector derived from the MinHash matrix for the provided vector.

    Example:
    minhash = np.array([[2, 3, 4], [5, 6, 7], [8, 9, 10]])
    vector = np.array([0, 1, 0])
    get_signature(minhash, vector)
    output:array([3, 6, 9])
    """
    idx = np.nonzero(vector)[0].tolist()
    shingles = minhash[:,idx]
    signature = np.min(shingles,axis=1)
    return signature

def jaccard_similarity(set1, set2):
    intersection_size = len(set1.intersection(set2))
    union_size = len(set1.union(set2))
    return intersection_size / union_size if union_size != 0 else 0.0

def compute_signature_similarity(signature_1, signature_2):
    """
    Calculate the similarity between two signature matrices using MinHash.

    Parameters:
    - signature_1: First signature matrix as a numpy array.
    - signature_matrix2: Second signature matrix as a numpy array.

    Returns:
    - Estimated Jaccard similarity.
    """
    # Ensure the matrices have the same shape
    if signature_1.shape != signature_2.shape:
        raise ValueError("Both signature matrices must have the same shape.")
    # Count the number of rows where the two matrices agree
    agreement_count = np.sum(signature_1 == signature_2)
    # Calculate the similarity
    similarity = agreement_count / signature_2.shape[0]

    return similarity


k = 2
shingle_sets = [shingle(text, k) for text in one_big_list]

vocab = build_vocab(shingle_sets)
print("Vocabulary size:", len(vocab))

one_hot_vectors = [one_hot(s, vocab) for s in shingle_sets]
one_hot_matrix = np.stack(one_hot_vectors)
print("One-hot matrix shape:", one_hot_matrix.shape)

num_hashes = 1000
minhash_arr = get_minhash_arr(num_hashes, vocab)

signatures = [get_signature(minhash_arr, vec) for vec in one_hot_vectors]
signatures = np.stack(signatures)
print("Signature matrix shape:", signatures.shape)

jaccard_sim = jaccard_similarity(shingle_sets[0], shingle_sets[1])
print("Jaccard similarity:", jaccard_sim)
minhash_sim = compute_signature_similarity(signatures[0], signatures[1])
print("MinHash similarity:", minhash_sim)

Vocabulary size: 1575
One-hot matrix shape: (4910, 1575)
Signature matrix shape: (4910, 1000)
Jaccard similarity: 0.38
MinHash similarity: 0.377


#### 6. Compute record similarity using LSH algorithm
  - Extract the top 2224 candidates from the LSH algorithm
  - Compare them to the actual mappings in the file DBLP-ACM_perfectMapping.csv
  - Compute the precision of the method.
  - Record the running time of the method.

In [26]:
start_time = time.time()

num_hashes = 1000
bands = 100
rows = 10

buckets = [{} for _ in range(bands)]
candidate_pairs = set()

for doc_id, signature in enumerate(signatures):
    for i in range(bands):
        start = i * rows
        end = start + rows
        band = tuple(signature[start:end])
        if band in buckets[i]:
            buckets[i][band].append(doc_id)
        else:
            buckets[i][band] = [doc_id]

for bucket_dict in buckets:
    for doc_ids in bucket_dict.values():
        if len(doc_ids) > 1:
            for pair in combinations(doc_ids, 2):
                candidate_pairs.add(tuple(sorted(pair)))

ranked_candidates = []
for p1, p2 in candidate_pairs:
    if (p1 < len(acm_df) and p2 >= len(acm_df)) or \
       (p2 < len(acm_df) and p1 >= len(acm_df)):
        sim = compute_signature_similarity(signatures[p1], signatures[p2])
        ranked_candidates.append((sim, (p1, p2)))

ranked_candidates.sort(key=lambda x: x[0], reverse=True)

top_k = 2224
candidates = {pair for sim, pair in ranked_candidates[:top_k]}

lsh_end_time = time.time()
lsh_runtime = lsh_end_time - start_time


acm_df['id'] = acm_df['id'].astype(str)
dblp2_df['id'] = dblp2_df['id'].astype(str)
dblpacm_df['idACM'] = dblpacm_df['idACM'].astype(str)
dblpacm_df['idDBLP'] = dblpacm_df['idDBLP'].astype(str)

acm_id_to_idx = {id_val: i for i, id_val in enumerate(acm_df['id'])}
dblp_id_to_idx = {id_val: i + len(acm_df) for i, id_val in enumerate(dblp2_df['id'])}

true_pairs = set()
for _, row in dblpacm_df.iterrows():
    acm_idx = acm_id_to_idx.get(row['idACM'])
    dblp_idx = dblp_id_to_idx.get(row['idDBLP'])

    if acm_idx is not None and dblp_idx is not None:
        pair = tuple(sorted((acm_idx, dblp_idx)))
        true_pairs.add(pair)

# Compute Precision
true_positives = len(candidates.intersection(true_pairs))
precision = true_positives / len(candidates) if candidates else 0


print(f"LSH generated {len(candidate_pairs)} initial candidate pairs.")
print(f"Selected top {len(candidates)} candidates for evaluation.")
print(f"True Positives: {true_positives}")
print(f"Precision: {precision:.4f}")
print(f"Running Time: {lsh_runtime:.2f} seconds")

LSH generated 7364 initial candidate pairs.
Selected top 2224 candidates for evaluation.
True Positives: 1984
Precision: 0.8921
Running Time: 0.80 seconds


#### 7. Compare the precision and the running time in Parts 1 and 2. 
- Pairwise comparison of all records:
    - Running time: 3048.07 seconds (~50 mins)
    - Precision: 0.92
- Finding similar records using LSH algorithm:
    - Running time: 0.80 seconds
    - Precision: 0.89

As we can see, the running time of the LSH algorithm is over 3 orders of magnitude smaller than that of the pairwise comparison, whereas the precision is still fairly high. 

# Task 3: Data preparation

In the cells below, we conduct the following data preparation steps on the [Pima Indians Diabetes Database](https://www.kaggle.com/uciml/pima-indians-diabetes-database):
1. Compute the correlation between the different columns after removing the outcome column.
2. Remove the disguised values from the table. We need to remove the values that equal to 0 from columns BloodPressure, SkinThickness and BMI as these are missing values but they have been replaced by the value 0. Remove the value but keep the record (i.e.) change the value to null.
3. Fill the cells with null using the mean values of the records that have the same class label.
4. Compute the correlation between the different columns.
5. Compare the values from this step with the values in the first step

In [27]:
df_diabetes = pd.read_csv("diabetes.csv", header = 0,
                 quotechar = '"',sep = ",",
                 na_values = ['na', '-', '.', ''])

In [28]:
df_diabetes

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


#### 1. Compute the correlation between the different columns after removing the outcome column.

In [29]:
# the columns method feteches all the column names except for "outcome" col, the outpur of columns is wrapped into the loc method to get all rows of resulting columns.
x = df_diabetes.loc[:,df_diabetes.columns[:-1]]
# corr() gives us the correlation matrix of the resulting columns.
x_matrix = x.corr()
x_matrix

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Pregnancies,1.000000,0.129459,0.141282,-0.081672,-0.073535,0.017683,-0.033523,0.544341
Glucose,0.129459,1.000000,0.152590,0.057328,0.331357,0.221071,0.137337,0.263514
BloodPressure,0.141282,0.152590,1.000000,0.207371,0.088933,0.281805,0.041265,0.239528
SkinThickness,-0.081672,0.057328,0.207371,1.000000,0.436783,0.392573,0.183928,-0.113970
Insulin,-0.073535,0.331357,0.088933,0.436783,1.000000,0.197859,0.185071,-0.042163
BMI,0.017683,0.221071,0.281805,0.392573,0.197859,1.000000,0.140647,0.036242
DiabetesPedigreeFunction,-0.033523,0.137337,0.041265,0.183928,0.185071,0.140647,1.000000,0.033561
Age,0.544341,0.263514,0.239528,-0.113970,-0.042163,0.036242,0.033561,1.000000


#### 2. Remove the disguised values from the table. 
We need to remove the values that equal to 0 from columns BloodPressure, SkinThickness and BMI as these are missing values but they have been replaced by the value 0. Remove the value but keep the record (i.e.) change the value to null.

In [30]:
#for each col mentioned in the cols list, replace() the 0 vals with nan.
cols = ['BloodPressure','SkinThickness','BMI']
for col in cols:
 df_diabetes[col] =  df_diabetes.loc[:,col].replace(0,np.nan)

#### 3. Fill the cells with null using the mean values of the records that have the same class label.

In [31]:
#the whole df is  effectively split into 2 groups, outcome 1 and 0, and then the mean of the column is calculated separately for each outcome. Here groupby() effectivey divides the df based on outcome
#the transform() fills the cell with the mean of each column separated for each outcome.
cols = df_diabetes.columns
df_diabetes[cols] = df_diabetes.groupby("Outcome")[cols].transform(lambda x: x.fillna(x.mean()))
df_diabetes

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35.0,0,33.6,0.627,50,1
1,1,85,66.0,29.0,0,26.6,0.351,31,0
2,8,183,64.0,33.0,0,23.3,0.672,32,1
3,1,89,66.0,23.0,94,28.1,0.167,21,0
4,0,137,40.0,35.0,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76.0,48.0,180,32.9,0.171,63,0
764,2,122,70.0,27.0,0,36.8,0.340,27,0
765,5,121,72.0,23.0,112,26.2,0.245,30,0
766,1,126,60.0,33.0,0,30.1,0.349,47,1


#### 4. Compute the correlation between the different columns.

In [32]:
y = df_diabetes.loc[:,df_diabetes.columns[:-1]]
y_corr = y.corr()
y_corr

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Pregnancies,1.000000,0.129459,0.208935,0.094172,-0.073535,0.024127,-0.033523,0.544341
Glucose,0.129459,1.000000,0.222417,0.220943,0.331357,0.219879,0.137337,0.263514
BloodPressure,0.208935,0.222417,1.000000,0.203453,-0.048106,0.286518,-0.002264,0.324439
SkinThickness,0.094172,0.220943,0.203453,1.000000,0.104017,0.565443,0.102426,0.135916
Insulin,-0.073535,0.331357,-0.048106,0.104017,1.000000,0.185545,0.185071,-0.042163
BMI,0.024127,0.219879,0.286518,0.565443,0.185545,1.000000,0.152530,0.027578
DiabetesPedigreeFunction,-0.033523,0.137337,-0.002264,0.102426,0.185071,0.152530,1.000000,0.033561
Age,0.544341,0.263514,0.324439,0.135916,-0.042163,0.027578,0.033561,1.000000


#### 5. Compare the values from this step with the values in the first step

In [33]:
x_df = pd.DataFrame(x.corr())
y_df = pd.DataFrame(y.corr())
print(x_df==y_df)

                          Pregnancies  Glucose  BloodPressure  SkinThickness  \
Pregnancies                      True     True          False          False   
Glucose                          True     True          False          False   
BloodPressure                   False    False           True          False   
SkinThickness                   False    False          False           True   
Insulin                          True     True          False          False   
BMI                             False    False          False          False   
DiabetesPedigreeFunction         True     True          False          False   
Age                              True     True          False          False   

                          Insulin    BMI  DiabetesPedigreeFunction    Age  
Pregnancies                  True  False                      True   True  
Glucose                      True  False                      True   True  
BloodPressure               False  False           